In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- datasets_read_csv ---
FIX_DATASETS_READ_CSV_BYTES = b',pol_num,status,issue_date,term_date\n0,101,active,2020-01-01,2020-12-31\n1,102,lapsed,2021-02-01,2021-12-31\n'
def FIX_DATASETS_READ_CSV_STREAM():
    return io.BytesIO(FIX_DATASETS_READ_CSV_BYTES)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_datasets_read_csv(stream):
    return pd.read_csv(stream,
                       index_col=0,
                       dtype={'pol_num': int,
                              'status': 'category'},
                       parse_dates=['issue_date', 'term_date'])
    return None

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_datasets_read_csv(stream):

    df = pl.read_csv(
        stream,
        schema_overrides={
            "pol_num": pl.Int64,
            "status": pl.Categorical,
        },
    )

    if df.width > 0:
        df = df.drop(df.columns[0])

    df = df.with_columns(
        [
            pl.col("issue_date").str.strptime(pl.Date, strict=False),
            pl.col("term_date").str.strptime(pl.Date, strict=False),
        ]
    )

    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: datasets_read_csv ===

# L1 smoke – generated
try:
    _r = gen_datasets_read_csv(FIX_DATASETS_READ_CSV_STREAM())
    print("✅ L1 smoke gen_datasets_read_csv: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_datasets_read_csv: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_datasets_read_csv(FIX_DATASETS_READ_CSV_STREAM())
    print("✅ L1 smoke before_datasets_read_csv: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_datasets_read_csv: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_datasets_read_csv(FIX_DATASETS_READ_CSV_STREAM())
    _rg = gen_datasets_read_csv(FIX_DATASETS_READ_CSV_STREAM())
    compare(_rb, _rg, "datasets_read_csv")
except Exception as _e:
    print(f"❌ L2 equivalence datasets_read_csv: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare a schema-valid header-only CSV on both sides.
try:
    import tempfile
    with tempfile.TemporaryDirectory() as _td:
        _p = Path(_td) / "empty.csv"
        _p.write_text(",pol_num,status,issue_date,term_date\n", encoding="utf-8")
        _rb = before_datasets_read_csv(_p)
        _rg = gen_datasets_read_csv(_p)
        compare(
            _rb.reset_index(drop=True), _rg,
            "L3 edge datasets_read_csv header only",
            check_row_order=True,
        )
except Exception as _e:
    print(f"❌ L3 edge datasets_read_csv: {type(_e).__name__}: {_e}")
